# TimeFolio contest notebook

This notebook does two things in one place:

1. Reads the latest local snapshot from `src/data/snapshots/latest.json` into a pandas DataFrame.
2. Provides browser-automation code for the `https://contest.timefolio.net/` order flow: login → `신규 주문` → `종목 선택` → `매수`/`매도` → `주문 비중` → `주문 제출`.

> Safety note: the automation helper defaults to `submit=False` so the order dialog is filled but not submitted unless you opt in.


## Optional one-time setup for browser automation

If Playwright is not installed in your notebook kernel, run:

```python
%pip install playwright
!python -m playwright install chromium
```


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Literal

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current working directory.')


def _repo_snapshot_relative_path(raw_path: str) -> Path | None:
    parts = Path(raw_path).parts
    marker = ('src', 'data', 'snapshots')
    for idx in range(len(parts) - len(marker) + 1):
        if parts[idx : idx + len(marker)] == marker:
            return Path(*parts[idx:])
    return None


def resolve_repo_snapshot_path(raw_path: str, repo_root: Path, run_id: str | None = None) -> Path:
    direct_path = Path(raw_path)
    if direct_path.exists():
        return direct_path.resolve()

    if not direct_path.is_absolute():
        repo_relative = (repo_root / direct_path).resolve()
        if repo_relative.exists():
            return repo_relative

    snapshot_relative = _repo_snapshot_relative_path(raw_path)
    if snapshot_relative is not None:
        local_path = (repo_root / snapshot_relative).resolve()
        if local_path.exists():
            return local_path

    if run_id:
        fallback = (repo_root / 'src' / 'data' / 'snapshots' / run_id / direct_path.name).resolve()
        if fallback.exists():
            return fallback

    raise FileNotFoundError(f'Could not resolve a local snapshot path for {raw_path!r}')


def read_latest_snapshot_manifest(repo_root: Path | None = None) -> tuple[Path, dict[str, Any]]:
    root = (repo_root or find_repo_root()).resolve()
    manifest_path = root / 'src' / 'data' / 'snapshots' / 'latest.json'
    with manifest_path.open('r', encoding='utf-8') as f:
        manifest = json.load(f)
    return manifest_path, manifest


def load_latest_snapshot_path(
    key: str = 'screened_stocks_by_score_json',
    *,
    repo_root: Path | None = None,
) -> tuple[Path, Path, dict[str, Any]]:
    root = (repo_root or find_repo_root()).resolve()
    manifest_path, manifest = read_latest_snapshot_manifest(root)
    snapshot_path = resolve_repo_snapshot_path(manifest[key], root, manifest.get('run_id'))
    return snapshot_path, manifest_path, manifest


REPO_ROOT = find_repo_root()
SNAPSHOT_PATH, SNAPSHOT_MANIFEST_PATH, SNAPSHOT_MANIFEST = load_latest_snapshot_path(repo_root=REPO_ROOT)
SNAPSHOT_PATH


In [ ]:
with SNAPSHOT_PATH.open('r', encoding='utf-8') as f:
    screened_stocks = json.load(f)

df = pd.json_normalize(screened_stocks)
sort_cols = [c for c in ['rank', 'final_score', 'stock_code'] if c in df.columns]
ascending = [True, False, True][: len(sort_cols)]
if sort_cols:
    df = df.sort_values(sort_cols, ascending=ascending, na_position='last').reset_index(drop=True)

print(
    f'Loaded {len(df):,} rows from {SNAPSHOT_PATH.relative_to(REPO_ROOT)} '
    f'(manifest: {SNAPSHOT_MANIFEST_PATH.relative_to(REPO_ROOT)})'
)
display(df.head(10))
df


In [ ]:
def pick_stock_candidate(frame: pd.DataFrame, rank: int = 1) -> pd.Series:
    """Return a single row that can be used as the default order candidate."""
    matches = frame.loc[frame['rank'] == rank] if 'rank' in frame.columns else frame
    if matches.empty:
        matches = frame
    return matches.iloc[0]


candidate = pick_stock_candidate(df)
candidate_fields = [c for c in ['rank', 'stock_code', 'stock_name', 'final_score'] if c in candidate.index]
display(candidate[candidate_fields])
candidate_query = str(candidate.get('stock_name') or candidate.get('stock_code'))
print('Suggested stock query:', candidate_query)


In [ ]:
def _require_playwright():
    try:
        from playwright.sync_api import TimeoutError as PlaywrightTimeoutError
        from playwright.sync_api import sync_playwright
    except ImportError as exc:
        raise RuntimeError(
            'Playwright is not installed. Run `%pip install playwright` and `python -m playwright install chromium` first.'
        ) from exc
    return sync_playwright, PlaywrightTimeoutError



def login_timefolio(page, email: str, password: str) -> None:
    page.goto('https://contest.timefolio.net/', wait_until='domcontentloaded')
    page.locator('#email').wait_for(state='visible', timeout=30_000)
    page.locator('#email').fill(email)
    page.locator('#password').fill(password)
    page.get_by_role('button', name='Submit').click()
    page.wait_for_load_state('networkidle')
    page.get_by_text('logout', exact=False).wait_for(timeout=30_000)


def open_order_form(page):
    page.get_by_role('button', name='신규 주문').click()
    form = page.locator('form').filter(has_text='주문 제출').first
    form.wait_for(state='visible', timeout=30_000)
    return form


def select_stock_in_form(page, form, stock_query: str) -> None:
    product_input = form.get_by_placeholder('종목 선택')
    product_input.click()
    product_input.fill(stock_query)
    page.wait_for_timeout(800)

    suggestion = page.locator('[role=option]').filter(has_text=stock_query).first
    if suggestion.count() and suggestion.is_visible():
        suggestion.click()
        return

    product_input.press('ArrowDown')
    product_input.press('Enter')


def set_order_side(form, side: Literal['buy', 'sell']) -> None:
    label = '매수' if side == 'buy' else '매도'
    choice = form.get_by_text(label, exact=True).first
    choice.click()


def set_order_weight(form, weight_pct: float) -> None:
    weight_input = form.locator('fieldset').filter(has_text='주문 비중').locator('input').first
    weight_input.click()
    weight_input.fill('')
    weight_input.type(f'{weight_pct}')


def place_contest_order(
    *,
    email: str,
    password: str,
    stock_query: str,
    side: Literal['buy', 'sell'],
    weight_pct: float,
    submit: bool = False,
    headless: bool = False,
    slow_mo_ms: int = 150,
    pause_before_close: bool = True,
) -> None:
    """
    Fill the contest order dialog using the site flow discovered from contest.timefolio.net.

    Parameters
    ----------
    submit:
        When False (default), the notebook fills the form and stops before clicking `주문 제출`.
        Set to True only when you want to send the order.
    """
    sync_playwright, PlaywrightTimeoutError = _require_playwright()

    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=headless, slow_mo=slow_mo_ms)
        page = browser.new_page(viewport={'width': 1440, 'height': 1200})
        try:
            login_timefolio(page, email=email, password=password)
            form = open_order_form(page)
            select_stock_in_form(page, form, stock_query=stock_query)
            set_order_side(form, side=side)
            set_order_weight(form, weight_pct=weight_pct)

            submit_button = form.get_by_role('button', name='주문 제출')
            if submit:
                submit_button.click()
                page.get_by_text('주문 추가 완료', exact=False).wait_for(timeout=15_000)
                print('Order submitted successfully.')
            else:
                print('Dry run complete. Review the filled dialog, then rerun with submit=True when ready.')
                if pause_before_close and not headless:
                    input('Press Enter to close the browser...')
        except PlaywrightTimeoutError as exc:
            raise RuntimeError('Timed out while waiting for the TimeFolio UI. Re-check selectors or login state.') from exc
        finally:
            browser.close()


In [ ]:
TIMEFOLIO_EMAIL = os.environ.get('TIMEFOLIO_EMAIL')
TIMEFOLIO_PASSWORD = os.environ.get('TIMEFOLIO_PASSWORD')

if not TIMEFOLIO_EMAIL or not TIMEFOLIO_PASSWORD:
    print('Set TIMEFOLIO_EMAIL and TIMEFOLIO_PASSWORD in the environment before running the automation cell.')
else:
    place_contest_order(
        email=TIMEFOLIO_EMAIL,
        password=TIMEFOLIO_PASSWORD,
        stock_query=candidate_query,
        side='buy',  # change to 'sell' when needed
        weight_pct=5.0,
        submit=False,  # set to True only when you really want to place the order
        headless=False,
    )
